# Rent Estimation Model Training

Reference notebook for the HomeLink AI Engine (`ai/` microservice).

## Goal
Train a scikit-learn pipeline that predicts monthly rent in ETB from:

- `area_sqm` (numeric)
- `bedrooms` (numeric)
- `bathrooms` (numeric)
- `has_water_tank` (numeric, 0/1)
- `subcity` (categorical, one-hot encoded)

## Training contract
The pipeline is saved to `../models/rent_estimator.joblib` and is loaded by `ai/src/rent_estimator.py`. Until a model is present, the service serves a deterministic heuristic baseline so it remains fully operational.

> **Feature order matters.** The serving module builds the numeric row `[area_sqm, bedrooms, bathrooms, has_water_tank]`. If the pipeline one-hot encodes `subcity`, those columns must be appended after the numeric columns at *training* time.

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

AI_DIR = Path.cwd().parent
MODELS_DIR = AI_DIR / "models"

NUMERIC_COLUMNS = ["area_sqm", "bedrooms", "bathrooms", "has_water_tank"]
CATEGORICAL_COLUMNS = ["subcity"]

In [ ]:
# Load a real dataset here (e.g. from the backend data pipeline).
# data = pd.read_csv("../notebooks/data/rent_listings.csv")

subcities = ["bole", "arada", "kirkos", "yeka", "gullele",
             "kolfe keranio", "nifas silk-lafto", "lideta",
             "addis ketema", "akaky kaliti"]
rng = np.random.default_rng(42)
n = 2_000
data = pd.DataFrame({
    "subcity": rng.choice(subcities, size=n),
    "bedrooms": rng.integers(1, 5, size=n),
    "bathrooms": rng.integers(1, 4, size=n),
    "area_sqm": rng.uniform(40, 200, size=n).round(1),
    "has_water_tank": rng.integers(0, 2, size=n),
})
subcity_rates = {"bole": 320, "arada": 300, "kirkos": 290, "yeka": 260,
                 "gullele": 240, "kolfe keranio": 230, "nifas silk-lafto": 225,
                 "lideta": 220, "addis ketema": 210, "akaky kaliti": 190}
data["rent_etb"] = (
    data["subcity"].map(subcity_rates) * data["area_sqm"]
    + data["bedrooms"] * 2500
    + data["bathrooms"] * 1500
    + data["has_water_tank"] * 1800
)
data["rent_etb"] = (data["rent_etb"] * rng.normal(1.0, 0.08, size=n)).round(0)

X = data[NUMERIC_COLUMNS + CATEGORICAL_COLUMNS]
y = data["rent_etb"]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_COLUMNS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLUMNS),
    ]
)
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("regressor", GradientBoostingRegressor(n_estimators=200, random_state=42)),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model.fit(X_train, y_train)

print("R2 (test):", round(model.score(X_test, y_test), 4))

In [ ]:
MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(model, MODELS_DIR / "rent_estimator.joblib")
print(f"Saved pipeline to {MODELS_DIR / 'rent_estimator.joblib'}")